# 02_crash_data_import.ipynb

## Traffic Accident Risk Prediction (TARP) – Week 1  
**Owner:** Khalid  
**Task:** Import and explore the Victorian Road Crash Data

### Objectives
- Download or load the Victorian Road Crash dataset
- Explore the dataset structure
- Inspect columns and basic data quality
- Identify useful variables for the capstone project
- Create a starter data dictionary


## 1. Dataset source

Official dataset: **Victoria road crash data** from the Victorian Government open data portal.  
For this notebook, we use the **accident.csv** table, which contains core accident-level records such as date, time, severity, crash type, road geometry, and speed zone.

> Note: The broader dataset is split across multiple related tables. The portal notes that the crash data is updated monthly with a 7‑month lag, and that location detail can be linked through node/location tables when needed later for hotspot mapping.


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)


## 2. Define file path

Update the path below if your CSV is stored in a different folder.

Suggested project structure:
```text
Traffic_Accident_Prediction/
├── data/
│   └── raw/
│       └── accident.csv
├── notebooks/
│   └── 02_crash_data_import.ipynb
└── outputs/
```


In [ ]:
# Option A: local file inside the repository
file_path = Path("../data/raw/accident.csv")

# Option B: use downloaded file in the current working directory
if not file_path.exists():
    file_path = Path("accident.csv")

print("Using file:", file_path.resolve())


## 3. Load the crash dataset

In [ ]:
df = pd.read_csv(file_path)
print("Dataset loaded successfully.")
print("Shape:", df.shape)
df.head()


## 4. Explore structure and columns

In [ ]:
df.info()


In [ ]:
columns_df = pd.DataFrame({
    "column_name": df.columns,
    "data_type": [str(df[col].dtype) for col in df.columns]
})
columns_df


## 5. Basic profiling

This section checks:
- missing values
- unique values
- sample ranges / categories


In [ ]:
profile_df = pd.DataFrame({
    "column_name": df.columns,
    "data_type": [str(df[col].dtype) for col in df.columns],
    "non_null_count": [df[col].notna().sum() for col in df.columns],
    "missing_count": [df[col].isna().sum() for col in df.columns],
    "missing_pct": [round(df[col].isna().mean() * 100, 2) for col in df.columns],
    "unique_values": [df[col].nunique(dropna=True) for col in df.columns]
})

profile_df.sort_values(by="missing_pct", ascending=False)


In [ ]:
print("Date range:", df["ACCIDENT_DATE"].min(), "to", df["ACCIDENT_DATE"].max())
print("Unique accident records:", df["ACCIDENT_NO"].nunique())
print("Duplicate ACCIDENT_NO:", df["ACCIDENT_NO"].duplicated().sum())


## 6. Quick look at important variables

In [ ]:
key_columns = [
    "ACCIDENT_DATE", "ACCIDENT_TIME", "ACCIDENT_TYPE_DESC", "DAY_WEEK_DESC",
    "DCA_DESC", "NODE_ID", "NO_OF_VEHICLES", "NO_PERSONS", "NO_PERSONS_KILLED",
    "NO_PERSONS_INJ_2", "NO_PERSONS_INJ_3", "ROAD_GEOMETRY_DESC", "SEVERITY",
    "SPEED_ZONE", "RMA"
]

df[key_columns].head(10)


In [ ]:
for col in ["ACCIDENT_TYPE_DESC", "DAY_WEEK_DESC", "ROAD_GEOMETRY_DESC", "SEVERITY", "SPEED_ZONE", "RMA"]:
    print(f"\n--- {col} ---")
    print(df[col].value_counts(dropna=False).head(15))


## 7. Identify useful variables for the project

For the Traffic Accident Risk Prediction project, the following variables are the most useful in the **accident** table:

### Core identifiers
- **ACCIDENT_NO** – unique accident record ID
- **NODE_ID** – useful for linking to location / node tables later

### Temporal variables
- **ACCIDENT_DATE**
- **ACCIDENT_TIME**
- **DAY_OF_WEEK / DAY_WEEK_DESC**

### Outcome / target-related variables
- **SEVERITY**
- **NO_PERSONS_KILLED**
- **NO_PERSONS_INJ_2**
- **NO_PERSONS_INJ_3**
- **NO_PERSONS**

### Crash context variables
- **ACCIDENT_TYPE / ACCIDENT_TYPE_DESC**
- **DCA_CODE / DCA_DESC**
- **ROAD_GEOMETRY / ROAD_GEOMETRY_DESC**
- **LIGHT_CONDITION**
- **SPEED_ZONE**
- **NO_OF_VEHICLES**
- **POLICE_ATTEND**
- **RMA**

### Variables likely less central for the first model
- duplicate description/code pairs can be reduced later
- some attributes may be kept only for interpretation, not modelling


## 8. Create a starter data dictionary

This is a practical project-level data dictionary for the variables most relevant to modelling.


In [ ]:
data_dictionary = pd.DataFrame([
    ["ACCIDENT_NO", "Unique identifier", "Primary key for each accident record", "Keep"],
    ["ACCIDENT_DATE", "Date", "Date of accident", "Keep"],
    ["ACCIDENT_TIME", "Time", "Time of accident", "Keep"],
    ["DAY_OF_WEEK", "Categorical / numeric code", "Day of week code", "Keep"],
    ["DAY_WEEK_DESC", "Categorical", "Day of week description", "Keep for interpretation"],
    ["ACCIDENT_TYPE", "Categorical / numeric code", "General crash type code", "Keep"],
    ["ACCIDENT_TYPE_DESC", "Categorical", "General crash type description", "Keep"],
    ["DCA_CODE", "Categorical / numeric code", "Definitions for Classifying Accidents code", "Keep"],
    ["DCA_DESC", "Categorical", "Detailed accident classification", "Keep"],
    ["LIGHT_CONDITION", "Categorical / numeric code", "Lighting condition at time of crash", "Keep"],
    ["NODE_ID", "Identifier", "Location linkage field for joining node/location table", "Keep"],
    ["NO_OF_VEHICLES", "Numeric", "Number of vehicles involved", "Keep"],
    ["NO_PERSONS_KILLED", "Numeric", "Number of persons killed", "Keep / derive severity"],
    ["NO_PERSONS_INJ_2", "Numeric", "Number of persons seriously injured", "Keep"],
    ["NO_PERSONS_INJ_3", "Numeric", "Number of persons otherwise injured", "Keep"],
    ["NO_PERSONS_NOT_INJ", "Numeric", "Number of persons not injured", "Optional"],
    ["NO_PERSONS", "Numeric", "Total persons involved", "Keep"],
    ["POLICE_ATTEND", "Categorical / numeric code", "Whether police attended", "Optional / useful"],
    ["ROAD_GEOMETRY", "Categorical / numeric code", "Road geometry code", "Keep"],
    ["ROAD_GEOMETRY_DESC", "Categorical", "Road geometry description", "Keep"],
    ["SEVERITY", "Ordinal / target candidate", "Crash severity category", "Keep"],
    ["SPEED_ZONE", "Categorical / numeric", "Posted speed zone", "Keep"],
    ["RMA", "Categorical", "Road management / road class", "Keep"]
], columns=["variable", "type", "description", "recommendation"])

data_dictionary


In [ ]:
# Save the data dictionary as CSV for the report / repository
output_path = Path("../outputs/data_dictionary_accident.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)

# If running outside the repo structure, fall back to local outputs folder
if not output_path.parent.exists():
    output_path = Path("outputs/data_dictionary_accident.csv")
    output_path.parent.mkdir(parents=True, exist_ok=True)

data_dictionary.to_csv(output_path, index=False)
print("Saved data dictionary to:", output_path.resolve())


## 9. Preliminary observations

### What we learned from the import step
- The accident table is at the **accident level**, with one row per crash.
- It contains strong **temporal**, **severity**, **road**, and **crash-context** features.
- **NODE_ID** will be important later for mapping and location-based analysis.
- **ACCIDENT_DATE** and **ACCIDENT_TIME** will allow feature engineering such as:
  - hour of day
  - weekday
  - weekend flag
  - rush hour
- **SEVERITY** can be used directly or converted into a binary target later.
- **RMA** may contain some missing values and should be checked in cleaning.

### Suggested next step
In Week 2, focus on:
- converting date/time to proper datetime format
- checking duplicate records
- cleaning categorical codes
- deciding how to define the modelling target


## 10. Deliverables completed
- Crash dataset imported
- Dataset columns explored
- Useful variables identified
- Starter data dictionary created
